# generals-bots transformer training (Colab GPU)

Runs `colab/train_colab.py` -- a standalone single-stage PPO loop for the transformer network (`training/network_transformer.py` + `training/augment.py`). See `~/.claude/plans/drifting-questing-babbage.md` Step 6 for the full design rationale. This does **not** run `training/train.py`'s full multi-stage curriculum -- it's a single fixed stage (the competition ruleset) against a Hunter/Expander/self-play mix, meant for an early real-GPU smoke test before committing to a long unattended run.

**One-time setup before running this notebook:**
1. Create a GitHub Personal Access Token (classic, `repo` scope is enough) at https://github.com/settings/tokens.
2. In this Colab notebook: the key (🔑) icon in the left sidebar → add a secret named `GITHUB_PAT` with that token as the value, and toggle notebook access on.
3. The repo URL below already defaults to your origin remote (`https://github.com/Amin-Debabeche/generals-bots`) -- change it if running against a fork.
4. **Optional, for live metrics charts:** create a Weights & Biases account at https://wandb.ai (free), grab your API key from https://wandb.ai/authorize, and add it as a Colab secret named `WANDB_API_KEY` the same way as the GitHub PAT above. Skip this if you're fine tailing `metrics.jsonl`/the cell output instead.

**Runtime:** Runtime menu → Change runtime type → GPU (T4 is fine for a first smoke test).

**Colab's free-tier session limits:** sessions disconnect after ~90 minutes idle or ~12 hours total. `train_colab.py` checkpoints to Drive every `--ckpt-every-minutes` (default 15, well inside that window) and every `--ckpt-every-iters` iterations, and resumes automatically from the last checkpoint on `--ckpt-dir` unless `--fresh` is passed -- so a disconnect just means re-running the cells below (with `--wandb`, it resumes logging to the SAME wandb run too, rather than starting a new one).

## 1. Mount Drive (checkpoints survive disconnects here)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

CKPT_DIR = '/content/drive/MyDrive/generals-bots-runs/colab1'  # change per run

## 2. Load secrets from Colab

`train_colab.py` itself never imports `google.colab` -- it just reads `$GITHUB_PAT`/`$WANDB_API_KEY` from the environment, so it stays runnable/testable outside Colab too. This cell is the only place secrets touch Colab-specific APIs. `WANDB_API_KEY` is optional -- leave the secret unset and this just silently skips it, no error.

In [ ]:
import os
from google.colab import userdata

os.environ['GITHUB_PAT'] = userdata.get('GITHUB_PAT')

try:
    os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY')
except Exception:
    pass  # no WANDB_API_KEY secret set -- fine, --wandb just won't be used below

## 3. Clone (or pull) the repo and install dependencies

In [ ]:
REPO_URL = 'https://github.com/Amin-Debabeche/generals-bots'  # change if running against a fork
REPO_DIR = '/content/generals-bots'

import os
if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git reset --hard && git clean -fd && git pull --rebase

%cd {REPO_DIR}
!pip install -q -e ".[training,colab]"

## 4. Confirm GPU + bf16 are actually visible to JAX

Worth checking before a long run -- see the plan's own note on this being a GPU/TPU-only design (bf16 has no benefit on CPU-only XLA).

In [ ]:
import jax
print(jax.devices())

## 5. Train

`--num-envs`/`--num-steps` start at the CNN path's own 512x256 default (see the plan's measured-memory note in Step 2) rather than AverageJoe's larger defaults -- scale up only once real GPU headroom on this box is confirmed, not guessed. `--minibatch-size` defaults to 128 in the script itself (much lower than PPOConfig's CNN-tuned 1024 -- a real run OOM'd at 1024 on a 15GB GPU); lower it further here if you still hit `RESOURCE_EXHAUSTED`.

`--push-status` commits a small `colab/status/<run-id>.json` back to the repo every `--status-every-iters` iterations, so a local Claude Code session (or you) can `git pull` and see progress without ever needing direct Colab access. `--wandb` additionally streams every iteration to a live Weights & Biases dashboard (loss/entropy/kl/reward/per-bucket win-rate charts, updating in real time) -- it's silently skipped if you didn't set the `WANDB_API_KEY` secret in step 2.

In [ ]:
!python -m colab.train_colab \
    --run-id colab1 \
    --ckpt-dir {CKPT_DIR} \
    --num-envs 512 --num-steps 256 \
    --max-wall-clock-hours 11 \
    --push-status --status-every-iters 10 \
    --wandb --wandb-project generals-bots-transformer

## Resuming after a disconnect

Re-run cells 1-4, then cell 5 unchanged (no `--fresh`) -- `train_colab.py` finds `train_meta.json` + `latest_full.eqx` under `--ckpt-dir` on Drive and picks up from the last saved iteration automatically.